In [1]:
import requests
import pandas as pd
import time
from io import StringIO

from config.settings import *

BASE = 'https://api.erg.ic.ac.uk/AirQuality'

def get_laqn_data(site_code, species, start, end):
    url = (f'{BASE}/Data/Site/SiteCode={site_code}'
           f'/StartDate={start}/EndDate={end}'
           f'/ResolutionHours=1/Species={species}/csv')
    r = requests.get(url, timeout=30)
    if r.status_code == 200 and len(r.content) > 200:
        return pd.read_csv(StringIO(r.text), skiprows=4)
    return None

# Step 1: Get all London monitoring sites
sites_url = f'{BASE}/Information/MonitoringSiteSpecies/GroupName=London/Json'
site_list = requests.get(sites_url).json()['Sites']['Site']
sites = pd.DataFrame(site_list)

print(f'Found {len(sites)} London monitoring sites')

# Step 2: Download data
frames = []

for _, row in sites.iterrows():
    for yr in range(2018, 2024):
        df = get_laqn_data(
            row['@SiteCode'],
            'NO2',
            f'{yr}-01-01',
            f'{yr}-12-31'
        )
        
        if df is not None:
            df['site_code'] = row['@SiteCode']
            df['borough'] = row.get('@LocalAuthorityName', 'Unknown')
            df['latitude'] = float(row.get('@Latitude', 0))
            df['longitude'] = float(row.get('@Longitude', 0))
            frames.append(df)
        
        time.sleep(0.3)

# Combine all data
laqn_raw = pd.concat(frames, ignore_index=True)

# Save locally
laqn_raw.to_parquet(f"{DATA_DIR}/laqn_raw.parquet")

print(f"Downloaded {len(laqn_raw):,} hourly readings")

ModuleNotFoundError: No module named 'config.settings'; 'config' is not a package